**BENCHMARKS**

This notebooks is to test our benchmark models and evaulate them on our test dataset. The models will be fit on the training data using the configs stored in utils/config.py

The available error metric are RMSE, MSE, MAE. By default we convert any predictions from raw transformed normalised level to log change and compute all metrics in the log change space. We can aggregate metric across channels, assets and horizons as needed.

The current available benchmarks are:
1. **Persistence** - here we predict raw data at each horizon for each asset for each channel to be equal to the last available data point in our window.
2. **Mean** - here we predict raw data at each horizon for each asset for each channel to be equal to the mean value for that asset for that channel over the context window.
3. **ARIMA** - here we predict each series (i.e. each channel for each asset) as a separate univariate ARIMA model. We can either fix this to be ARIMA(1,0,1) or use auto arima to automatically select the order. These models are fit on log return data.
4. **VAR** - here we predict our data using a VAR that simulataneously uses all of the data for a given channel across all assets and models them jointly. We can automatically infer the optimal value of the p in VAR(p). 

In [47]:
from pathlib import Path
import sys
import torch
import pandas as pd
# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits, get_channel
from src.evaluation.metrics import mae, rmse
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.utils.config import load_yaml
from src.utils.metric_tables import make_metric_table

Project root: /Users/vishalruparelia/Desktop/Thesis/dynamic_graphs_thesis


In [23]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

Load the data and clean

In [24]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

train samples: 187
val samples: 42
test samples: 20
channels: ['open', 'high', 'low', 'close', 'volume', 'amount']
assets: 93
stride: 15
input features: ['open', 'high', 'low', 'close', 'volume', 'amount']
targets: ['open', 'high', 'low', 'close']


**Persistence**

In [26]:
persistence = PersistenceBaseline.from_config(config)

persistence.fit(
    train_split=train,
    val_split=val,
)

persistence_result = persistence.predict(
    split=test,
    output_space="cumulative_log_change",
    batch_size=256,
)

metric = "MAE"
persistence_metric_table = make_metric_table(
    metric=metric,
    y_pred=persistence_result["y_pred"],
    y_true=persistence_result["y_true"],
    horizons=persistence_result["horizons"],
    channels=persistence_result["channels"],
    assets=test["asset_cols"],
    horizon=[1,5,15,30,60],
    channel=['close','high','low','open'],
)

persistence_metric_table.pivot(index="horizon",
    columns="channel",
    values=metric.upper(),
)

channel,close,high,low,open
horizon,,,,
1,0.000386,0.000338,0.000337,0.000370
5,0.000824,0.000814,0.000819,0.000838
15,0.001364,0.001361,0.001366,0.001376
30,0.001868,0.001871,0.001878,0.001894
60,0.002693,0.002695,0.002711,0.002716


**Mean**

In [8]:
mean = MeanBaseline.from_config(config)

mean.fit(
    train_split = train,
    val_split = val
)

mean_result = mean.predict(
    split = test,
    output_space = 'cumulative_log_change',
    batch_size = 256
)

metric = "MAE"
mean_metric_table = make_metric_table(
    metric=metric,
    y_pred=mean_result["y_pred"],
    y_true=mean_result["y_true"],
    horizons=mean_result["horizons"],
    channels=mean_result["channels"],
    assets=test["asset_cols"],
    horizon=[1,5,15,30,60],
    channel=['close','high','low','open'],
)

mean_metric_table.pivot(index="horizon",
    columns="channel",
    values=metric.upper(),
)

channel,close,high,low,open
horizon,,,,
1,0.001686,0.001679,0.001689,0.001685
5,0.001845,0.001841,0.001848,0.001850
15,0.002139,0.002138,0.002146,0.002149
30,0.002531,0.002527,0.002539,0.002539
60,0.003207,0.003200,0.003225,0.003218


**ARIMA**

In [ ]:
config = load_yaml(CONFIG_PATH)

arima = ArimaBaseline.from_config(
    config,
    fit_mode="auto",
    optim_method='powell',
)

arima.fit(
    train_split=train,
    val_split=val,
)

arima_result = arima.predict(
    split=test,
    output_space="cumulative_log_change",
)

metric = "MAE"
arima_metric_table = make_metric_table(
    metric=metric,
    y_pred=arima_result["y_pred"],
    y_true=arima_result["y_true"],
    horizons=arima_result["horizons"],
    channels=arima_result["channels"],
    assets=test["asset_cols"],
    horizon=[1,5,15,30,60],
    channel=['close'],
)

arima_metric_table.pivot(index="horizon",
    columns="channel",
    values=metric.upper(),
)

In [38]:
arima_metric_table.pivot(index="horizon",
    columns="channel",
    values=metric.upper(),
)

channel,close
horizon,
1,0.000387
5,0.000824
15,0.001365
30,0.001870
60,0.002700


In [21]:
order_table = (
    pd.Series(arima_result['selected_orders'].values())
    .value_counts()
    .rename_axis("order")
    .reset_index(name="count")
)

order_table

,order,count
0,"(0, 0, 1)",20
1,"(0, 0, 0)",14
2,"(1, 0, 0)",13
3,"(1, 0, 1)",10
4,"(0, 0, 2)",9
5,"(2, 0, 0)",8
6,"(3, 0, 0)",6
7,"(2, 0, 1)",3
8,"(1, 0, 2)",3
9,"(1, 0, 3)",2


**VAR**

In [49]:
config = load_yaml(CONFIG_PATH)

var = VarBaseline.from_config(
    config,
    maxlags=15,
    ic="aic",
    trend="c",
)

var.fit(
    train_split=train,
    val_split=val,
)

var_result = var.predict(
    split=test,
    output_space="cumulative_log_change",
)

metric = "MAE"
var_metric_table = make_metric_table(
    metric=metric,
    y_pred=var_result["y_pred"],
    y_true=var_result["y_true"],
    horizons=var_result["horizons"],
    channels=var_result["channels"],
    assets=test["asset_cols"],
    horizon=[1, 5, 15, 30, 60],
    channel=['close','high','low','open'],
)

var_metric_table.pivot(
    index="horizon",
    columns="channel",
    values=metric.upper(),
)

Fitting 4 VAR model(s) with maxlags=15, ic=aic...
  open: selected_lag=11, failed=False
  high: selected_lag=11, failed=False
  low: selected_lag=10, failed=False
  close: selected_lag=11, failed=False
Finished fitting VAR models.
Failed models: 0


channel,close,high,low,open
horizon,,,,
1,0.000397,0.000347,0.000345,0.000379
5,0.000837,0.000825,0.000828,0.000849
15,0.001377,0.001370,0.001376,0.001388
30,0.001877,0.001875,0.001886,0.001899
60,0.002703,0.002692,0.002729,0.002722
